# ARC_ATLAS v4 - Native-Geometry nnU-Net

This notebook uses the same training split as the recent v4 runs: `data/splits/90_10_random/train/manifest.csv` with ATLAS, ARC, and Approx cases.

The nnU-Net folders created here are only a format view required by nnU-Net: images are symlinked from the same split and labels are written as clean `uint8` masks. The source MRIs keep their native full dimensions and affine/spacing. No `TARGET_SHAPE` resize is applied here.

Training is `3d_fullres`. nnU-Net will try to keep full voxel resolution and choose the largest 3D patch that fits GPU memory. A true whole-volume tensor is only feasible if the planner/GPU can fit it; otherwise patch-based training is the memory fallback, while still using full-resolution MRIs.

In [16]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4").resolve()
os.chdir(PROJECT_ROOT)

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import atlas_nnunet_pipeline as pipe

SOURCE_DATA_DIR = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "train"
MANIFEST = SOURCE_DATA_DIR / "manifest.csv"
NNUNET_VIEW_ROOT = SOURCE_DATA_DIR / "nnunet_view"
CV_ROOT = PROJECT_ROOT / "runs" / "nnunet_cv_predictions_native_split"
PLANS_NAME = "nnUNetPlans_24GB"

layout = pipe.NnUNetLayout(
    project_root=PROJECT_ROOT,
    nnunet_root=NNUNET_VIEW_ROOT,
    dataset_id=701,
    dataset_name="ARC_ATLAS_TrainV4Native",
)

print("Project:", PROJECT_ROOT)
print("Source dataset:", SOURCE_DATA_DIR)
print("Manifest:", MANIFEST)
print("nnU-Net view raw:", layout.raw)
print("nnU-Net view preprocessed:", layout.preprocessed)
print("nnU-Net view results:", layout.results)

Project: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4
Source dataset: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train
Manifest: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv
nnU-Net view raw: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_raw
nnU-Net view preprocessed: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed
nnU-Net view results: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_results


## 1. Runtime Check

In [17]:
pipe.print_preflight(layout)

checks = pipe.preflight(layout)
missing = [k for k, v in checks.items() if str(v).startswith("missing") or (k.startswith("nnUNetv2_") and not v)]
if missing:
    print("\nMissing runtime pieces:", missing)
    print("Install in tf_310 with: pip install -r requirements_nnunet.txt")

python                           /home/rbielski/miniconda3/envs/tf_310/bin/python
nnUNet_raw                       /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_raw
nnUNet_preprocessed              /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed
nnUNet_results                   /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_results
nnUNetv2_plan_and_preprocess     /home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_plan_and_preprocess
nnUNetv2_train                   /home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_train
nnUNetv2_predict                 /home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_predict
python:torch                     2.5.1+cu124
python:nnunetv2                  installed
python:nibabel                   5.3.2
python:scipy               

## 2. Inspect The Exact Source Split

This is the dataset choice. It should match the v4 split you were using before.

In [18]:
import pandas as pd
import nibabel as nib

source_df = pd.read_csv(MANIFEST)
print("Rows:", len(source_df))
display(source_df.groupby("slug").size().rename("n").to_frame())

shape_rows = []
for _, row in source_df.iterrows():
    img = nib.load(row.t1)
    shape_rows.append({
        "slug": row.slug,
        "shape": "x".join(map(str, img.shape)),
        "spacing": "x".join(f"{float(z):.6g}" for z in img.header.get_zooms()[:3]),
    })
shape_df = pd.DataFrame(shape_rows)
print("Native image shapes / spacing in the source split:")
display(shape_df.groupby(["slug", "shape", "spacing"]).size().rename("n").reset_index().sort_values(["slug", "n"], ascending=[True, False]))

Rows: 866


,n
slug,
ARC-combined-t1-raw-ab0d1794,190
ATLAS-Images-f0d7431e,582
Approx-Numeracy-Processed,94


Native image shapes / spacing in the source split:


,slug,shape,spacing,n
0,ARC-combined-t1-raw-ab0d1794,193x229x193,1x1x1,190
1,ATLAS-Images-f0d7431e,193x229x193,1x1x1,582
2,Approx-Numeracy-Processed,193x229x193,1x1x1,94


## 3. Build nnU-Net View Of The Same Data

This does not change the source dataset. It creates nnU-Net's required `imagesTr`/`labelsTr` layout under the same split directory, symlinking MRIs from `t1/` and writing binary label files from `masks/`.

In [19]:
OVERWRITE_NNUNET_VIEW = False

if not layout.mapping_csv.exists() or OVERWRITE_NNUNET_VIEW:
    summary = pipe.prepare_nnunet_dataset(
        MANIFEST,
        layout,
        copy_mode="symlink",
        overwrite=OVERWRITE_NNUNET_VIEW,
        n_splits=5,
    )
else:
    with (layout.raw_dataset_dir / "conversion_summary.json").open() as f:
        summary = json.load(f)

print(json.dumps(summary, indent=2))

{
  "source_manifest": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/manifest.csv",
  "dataset_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_raw/Dataset701_ARC_ATLAS_TrainV4Native",
  "preprocessed_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed/Dataset701_ARC_ATLAS_TrainV4Native",
  "results_dir": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_results",
  "dataset_id": 701,
  "dataset_name": "Dataset701_ARC_ATLAS_TrainV4Native",
  "num_cases": 866,
  "copy_mode": "symlink",
  "folds": [
    {
      "fold": 0,
      "train": 692,
      "val": 174
    },
    {
      "fold": 1,
      "train": 693,
      "val": 173
    },
    {
      "fold": 2,
      "train": 693,
      "val": 173
    },
    {
     

In [20]:
mapping = pd.read_csv(layout.mapping_csv)
print("Cases in nnU-Net view:", len(mapping))
display(mapping.groupby(["source_slug", "fold"]).size().unstack(fill_value=0))
display(mapping.groupby(["fold", "lesion_group"]).size().unstack(fill_value=0))
display(mapping[["case_id", "fold", "source_slug", "lesion_voxels", "shape", "spacing", "key"]].head())

Cases in nnU-Net view: 866


fold,0,1,2,3,4
source_slug,,,,,
ARC-combined-t1-raw-ab0d1794,37,37,35,40,41
ATLAS-Images-f0d7431e,115,115,124,113,115
Approx-Numeracy-Processed,22,21,14,20,17


lesion_group,000001_000099,000100_000999,001000_009999,010000_plus
fold,,,,
0,6,36,46,86
1,5,35,47,86
2,5,35,47,86
3,5,35,47,86
4,5,35,47,86


,case_id,fold,source_slug,lesion_voxels,shape,spacing,key
0,case_0001_sub_m2306,3,ARC-combined-t1-raw-ab0d1794,149459,193x229x193,1x1x1,sub-M2306_ses-707_acq-tfl3p2_run-3_T1w_MNI_nor...
1,case_0002_sub_r034s039,3,ATLAS-Images-f0d7431e,51687,193x229x193,1x1x1,sub-r034s039_ses-1_space-MNI152NLin2009aSym_T1...
2,case_0003_sub_062,0,Approx-Numeracy-Processed,45373,193x229x193,1x1x1,sub-062_T1w_MNI_norm.nii.gz
3,case_0004_sub_r011s017,3,ATLAS-Images-f0d7431e,823,193x229x193,1x1x1,sub-r011s017_ses-1_space-MNI152NLin2009aSym_T1...
4,case_0005_sub_m2283,4,ARC-combined-t1-raw-ab0d1794,48959,193x229x193,1x1x1,sub-M2283_ses-246_acq-tfl3p2_run-4_T1w_MNI_nor...


## 4. Plan At Full Resolution First

Run the `--no-preprocess` cells first to inspect patch size before preprocessing. The default plan is conservative. The 24 GB plan targets this machine's RTX 4090 memory and gets closer to the full volume, but nnU-Net still crops nonzero background before planning unless we add a custom no-crop preprocessor.

Raw source shape is `193x229x193`; current nnU-Net crop median is `156x186x148`.

In [21]:
pipe.print_recommended_commands(layout)

# Environment
export nnUNet_raw=/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_raw
export nnUNet_preprocessed=/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed
export nnUNet_results=/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_results

# Plan and preprocess
nnUNetv2_plan_and_preprocess -d 701 --verify_dataset_integrity -c 3d_fullres

# Train five full-resolution 3D folds
nnUNetv2_train 701 3d_fullres 0 --npz
nnUNetv2_train 701 3d_fullres 1 --npz
nnUNetv2_train 701 3d_fullres 2 --npz
nnUNetv2_train 701 3d_fullres 3 --npz
nnUNetv2_train 701 3d_fullres 4 --npz

# Leak-free CV inference
/home/rbielski/miniconda3/envs/tf_310/bin/python src/atlas_nnunet_pipeline.py predict-cv --nnunet-root /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90

In [22]:
RUN_FINGERPRINT_AND_PLAN_ONLY = True

if RUN_FINGERPRINT_AND_PLAN_ONLY:
    pipe.plan_and_preprocess(layout, verify=True, no_preprocess=True, num_processes=[4])
else:
    print("Set RUN_FINGERPRINT_AND_PLAN_ONLY=True to verify data and inspect default 3d_fullres patch size without preprocessing.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_plan_and_preprocess -d 701 --verify_dataset_integrity --no_pp -c 3d_fullres -np 4
Fingerprint extraction...
Dataset701_ARC_ATLAS_TrainV4Native
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [156. 186. 148.], 3d_lowres: [156, 186, 148]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 

In [23]:
RUN_24GB_PLAN_ONLY = True

if RUN_24GB_PLAN_ONLY:
    pipe.plan_and_preprocess(
        layout,
        verify=False,
        no_preprocess=True,
        configurations=("3d_fullres",),
        num_processes=[4],
        gpu_memory_target=24,
        overwrite_plans_name="nnUNetPlans_24GB",
    )
else:
    print("Set RUN_24GB_PLAN_ONLY=True to create a 24GB planning-only variant before preprocessing.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_plan_and_preprocess -d 701 --no_pp -c 3d_fullres -np 4 -gpu_memory_target 24 -overwrite_plans_name nnUNetPlans_24GB
Fingerprint extraction...
Dataset701_ARC_ATLAS_TrainV4Native
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [156. 186. 148.], 3d_lowres: [156, 186, 148]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_24GB_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 318, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([186., 148.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_no

In [24]:
RUN_PLAN_AND_PREPROCESS = True

if RUN_PLAN_AND_PREPROCESS:
    pipe.plan_and_preprocess(
        layout,
        verify=True,
        configurations=("3d_fullres",),
        num_processes=[4],
        gpu_memory_target=24,
        overwrite_plans_name=PLANS_NAME,
    )
else:
    print("Set RUN_PLAN_AND_PREPROCESS=True after the planner patch size looks acceptable.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_plan_and_preprocess -d 701 --verify_dataset_integrity -c 3d_fullres -np 4 -gpu_memory_target 24 -overwrite_plans_name nnUNetPlans_24GB
Fingerprint extraction...
Dataset701_ARC_ATLAS_TrainV4Native
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [156. 186. 148.], 3d_lowres: [156, 186, 148]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_24GB_2d', 

Preprocessing cases: 100%|██████████| 866/866 [06:41<00:00,  2.16it/s]


In [25]:
RUN_FINGERPRINT_AND_PLAN_ONLY = True

if RUN_FINGERPRINT_AND_PLAN_ONLY:
    pipe.plan_and_preprocess(layout, verify=True, no_preprocess=True, num_processes=[4])
else:
    print("Set RUN_FINGERPRINT_AND_PLAN_ONLY=True to verify data and inspect default 3d_fullres patch size without preprocessing.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_plan_and_preprocess -d 701 --verify_dataset_integrity --no_pp -c 3d_fullres -np 4
Fingerprint extraction...
Dataset701_ARC_ATLAS_TrainV4Native
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [156. 186. 148.], 3d_lowres: [156, 186, 148]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 

## 5. Train Five Native-Split Folds

In [26]:
RUN_TRAINING = True
TRAIN_FOLDS = [0, 1, 2, 3, 4]

if RUN_TRAINING:
    for fold in TRAIN_FOLDS:
        pipe.train_fold(layout, fold, configuration="3d_fullres", plans=PLANS_NAME, save_npz=True)
else:
    print("Set RUN_TRAINING=True to train the five 3d_fullres folds.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_train 701 3d_fullres 0 -p nnUNetPlans_24GB --npz
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-01 15:32:33.557843: Using torch.compile...
2026-05-01 15:32:34.139784: do_dummy_2d_data_aug: False
2026-05-01 15:32:34.141302: Using splits from existing split file: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/90_10_random/train/nnunet_view/nnUNet_preprocessed/Dataset701_ARC_ATLAS_TrainV4Native/splits_final.json
2026-05-01 15:32:34.141662: The split file contains 5 splits.
2026-05-01 15:32:34.141701: Desired fold for training: 0

## 6. Leak-Free CV Prediction, Postprocess, Evaluate

In [27]:
RUN_CV_PREDICT = True

if RUN_CV_PREDICT:
    pipe.predict_cv_folds(layout, CV_ROOT, configuration="3d_fullres", plans=PLANS_NAME)
else:
    print("Set RUN_CV_PREDICT=True after all requested folds have checkpoints.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_predict -i /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_0_input -o /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_0_pred -d 701 -c 3d_fullres -f 0 -p nnUNetPlans_24GB --save_probabilities

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 174 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 174 cases that I would like to predict

Predicting case_0003_sub_062:
perform_everything_on_devic

  0%|          | 0/1 [00:00<?, ?it/s]

sending off prediction to background worker for resampling and export
done with case_0003_sub_062

Predicting case_0008_sub_r031s032:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.22it/s]


sending off prediction to background worker for resampling and export
done with case_0008_sub_r031s032

Predicting case_0009_sub_m2277:
perform_everything_on_device: True


...

sending off prediction to background worker for resampling and export
done with case_0845_sub_r001s037

Predicting case_0852_sub_m2144:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.21it/s]


sending off prediction to background worker for resampling and export
done with case_0852_sub_m2144

Predicting case_0857_sub_m2164:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.21it/s]


sending off prediction to background worker for resampling and export
done with case_0857_sub_m2164


In [28]:
RUN_CV_POSTPROCESS = True

if RUN_CV_POSTPROCESS:
    post_dirs = []
    for pred_dir in sorted(CV_ROOT.glob("fold_*_pred")):
        out_dir = pred_dir.with_name(pred_dir.name + "_postprocessed")
        pipe.postprocess_probabilities(pred_dir, out_dir, mapping_csv=layout.mapping_csv)
        post_dirs.append(out_dir)
    print("Postprocessed:", post_dirs)
else:
    print("Set RUN_CV_POSTPROCESS=True if CV predictions were saved with probabilities.")

Postprocessed: [PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_0_pred_postprocessed'), PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_1_pred_postprocessed'), PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_2_pred_postprocessed'), PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_3_pred_postprocessed'), PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/fold_4_pred_postprocessed')]


In [29]:
def has_prediction_masks(path):
    return path.is_dir() and any(path.glob("*.nii.gz"))

raw_pred_dirs = [d for d in sorted(CV_ROOT.glob("fold_*_pred")) if has_prediction_masks(d)]
post_pred_dirs = [d for d in sorted(CV_ROOT.glob("fold_*_pred_postprocessed")) if has_prediction_masks(d)]
eval_dirs = post_pred_dirs or raw_pred_dirs

if eval_dirs:
    summary = pipe.evaluate_predictions(eval_dirs, layout.mapping_csv, CV_ROOT / "evaluation.csv")
    print(json.dumps(summary, indent=2))
else:
    print("No CV prediction masks found yet:", CV_ROOT)

{
  "n": 866,
  "dice_mean": 0.022912144699769054,
  "dice_median": 0.0,
  "dice_by_group": {
    "000001_000099": {
      "n": 26,
      "mean": 0.006423811153846154,
      "median": 0.0
    },
    "000100_000999": {
      "n": 176,
      "mean": 0.01221007318181818,
      "median": 0.0
    },
    "001000_009999": {
      "n": 234,
      "mean": 0.019669617863247866,
      "median": 0.0
    },
    "010000_plus": {
      "n": 430,
      "mean": 0.030054034325581395,
      "median": 0.0
    }
  },
  "output_csv": "/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_cv_predictions_native_split/evaluation.csv"
}


## 7. All-Fold Inference For External/Test Data

In [32]:
RUN_TEST_PREDICT = True
TEST_MANIFEST = PROJECT_ROOT / "data" / "splits" / "90_10_random" / "test" / "manifest.csv"
TEST_INPUT = PROJECT_ROOT / "runs" / "nnunet_test_input_native"
TEST_PRED = PROJECT_ROOT / "runs" / "nnunet_test_predictions_native"
TEST_POST = PROJECT_ROOT / "runs" / "nnunet_test_predictions_native_postprocessed"

if RUN_TEST_PREDICT:
    if not TEST_MANIFEST.exists():
        raise FileNotFoundError(f"No test manifest found: {TEST_MANIFEST}")
    pipe.prepare_prediction_input_from_manifest(TEST_MANIFEST, TEST_INPUT, project_root=PROJECT_ROOT, overwrite=True)
    pipe.predict(layout, TEST_INPUT, TEST_PRED, folds=(0, 1, 2, 3, 4), configuration="3d_fullres", plans=PLANS_NAME)
    pipe.postprocess_probabilities(TEST_PRED, TEST_POST, mapping_csv=TEST_INPUT / "case_mapping.csv")
    print("Postprocessed predictions:", TEST_POST)
else:
    print("Set RUN_TEST_PREDICT=True after all five folds are trained.")

/home/rbielski/miniconda3/envs/tf_310/bin/nnUNetv2_predict -i /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_test_input_native -o /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_test_predictions_native -d 701 -c 3d_fullres -f 0 1 2 3 4 -p nnUNetPlans_24GB --save_probabilities

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 96 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 96 cases that I would like to predict

Predicting case_0001_sub_m2050:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.37it/s]


sending off prediction to background worker for resampling and export
done with case_0001_sub_m2050

Predicting case_0002_sub_r048s029:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.37it/s]


sending off prediction to background worker for resampling and export
done with case_0002_sub_r048s029

Predicting case_0003_sub_r031s006:
perform_everything_on_device: True


...

sending off prediction to background worker for resampling and export
done with case_0094_sub_r034s032

Predicting case_0095_sub_r028s020:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.37it/s]


sending off prediction to background worker for resampling and export
done with case_0095_sub_r028s020

Predicting case_0096_sub_r047s001:
perform_everything_on_device: True


100%|██████████| 1/1 [00:00<00:00,  5.37it/s]


sending off prediction to background worker for resampling and export
done with case_0096_sub_r047s001
Postprocessed predictions: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/nnunet_test_predictions_native_postprocessed
